# 03 — A/B Test Retention Analysis

**Objective:** Determine whether the new recommendation module (gate_40) significantly improves retention compared to the control (gate_30).

**Primary metric:** 7-day retention (`retention_7`)  
**Secondary metric:** 1-day retention (`retention_1`)  

**Steps in this notebook:**
1. Calculate retention rates for each group
2. Compute absolute and relative uplift
3. Run hypothesis tests (two-proportion z-test)
4. Compute 95% confidence intervals
5. Visualise results

**Input:** `../data/app_ab_test.db`

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Connect to database
# 连接数据库
conn = sqlite3.connect('../data/app_ab_test.db')
df = pd.read_sql_query('SELECT * FROM users', conn)

print(f'Loaded {len(df):,} users')
df.head()

Loaded 90,189 users


,userid,version,treatment,sum_gamerounds,retention_1,retention_7,engagement_segment
0,116,gate_30,0,3,0,0,low
1,337,gate_30,0,38,1,0,high
2,377,gate_40,1,165,1,0,high
3,483,gate_40,1,1,0,0,low
4,488,gate_40,1,179,1,1,high


---
## 1. Retention Rates by Group
计算两组的留存率

In [2]:
# Split into control and treatment groups
# 拆分成 control 和 treatment 两组
ctrl = df[df['version'] == 'gate_30']
trt  = df[df['version'] == 'gate_40']

# Count users in each group
# 统计每组用户数
n_ctrl = len(ctrl)
n_trt  = len(trt)

# Count retained users
# 统计留存用户数
retained_d1_ctrl = ctrl['retention_1'].sum()   # 第1天回来的 control 用户数
retained_d1_trt  = trt['retention_1'].sum()    # 第1天回来的 treatment 用户数
retained_d7_ctrl = ctrl['retention_7'].sum()   # 第7天回来的 control 用户数
retained_d7_trt  = trt['retention_7'].sum()    # 第7天回来的 treatment 用户数

# Calculate retention rates
# 计算留存率 = 留存用户数 / 总用户数
rate_d1_ctrl = retained_d1_ctrl / n_ctrl
rate_d1_trt  = retained_d1_trt  / n_trt
rate_d7_ctrl = retained_d7_ctrl / n_ctrl
rate_d7_trt  = retained_d7_trt  / n_trt

print('=== Retention Rates ===')
print(f'                     Control (gate_30)    Treatment (gate_40)')
print(f'Users:               {n_ctrl:>10,}         {n_trt:>10,}')
print(f'D1 retained:         {retained_d1_ctrl:>10,}         {retained_d1_trt:>10,}')
print(f'D1 retention rate:   {rate_d1_ctrl:>10.1%}         {rate_d1_trt:>10.1%}')
print(f'D7 retained:         {retained_d7_ctrl:>10,}         {retained_d7_trt:>10,}')
print(f'D7 retention rate:   {rate_d7_ctrl:>10.1%}         {rate_d7_trt:>10.1%}')

=== Retention Rates ===
                     Control (gate_30)    Treatment (gate_40)
Users:                   44,700             45,489
D1 retained:             20,034             20,119
D1 retention rate:        44.8%              44.2%
D7 retained:              8,502              8,279
D7 retention rate:        19.0%              18.2%


In [3]:
# Calculate uplift
# 计算提升幅度
# Absolute uplift = treatment rate - control rate
# 绝对提升 = treatment 留存率 - control 留存率
# Relative uplift = absolute uplift / control rate
# 相对提升 = 绝对提升 / control 留存率

abs_uplift_d1 = rate_d1_trt - rate_d1_ctrl
rel_uplift_d1 = abs_uplift_d1 / rate_d1_ctrl

abs_uplift_d7 = rate_d7_trt - rate_d7_ctrl
rel_uplift_d7 = abs_uplift_d7 / rate_d7_ctrl

print('=== Uplift Summary ===')
print(f'                D1 Retention    D7 Retention (Primary)')
print(f'Control:        {rate_d1_ctrl:.3%}         {rate_d7_ctrl:.3%}')
print(f'Treatment:      {rate_d1_trt:.3%}         {rate_d7_trt:.3%}')
print(f'Absolute uplift:{abs_uplift_d1:+.3%}         {abs_uplift_d7:+.3%}')
print(f'Relative uplift:{rel_uplift_d1:+.2%}         {rel_uplift_d7:+.2%}')
print()
print('Note: negative uplift means treatment performed WORSE than control.')
print('注意：负的 uplift 表示 treatment 组表现比 control 组更差。')

=== Uplift Summary ===
                D1 Retention    D7 Retention (Primary)
Control:        44.819%         19.020%
Treatment:      44.228%         18.200%
Absolute uplift:-0.591%         -0.820%
Relative uplift:-1.32%         -4.31%

Note: negative uplift means treatment performed WORSE than control.
注意：负的 uplift 表示 treatment 组表现比 control 组更差。
